In [ ]:
import os
import re
import subprocess
import sys

try:
    import reportlab  # noqa: F401
except ImportError:
    print("reportlab not found -- attempting to install it now "
          "(requires internet access to be enabled in Kaggle's notebook settings)...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "reportlab"])
    print("reportlab installed successfully.")

import openpyxl
from reportlab.lib import colors
from reportlab.lib.enums import TA_CENTER, TA_LEFT
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import ParagraphStyle
from reportlab.platypus import (
    Paragraph, SimpleDocTemplate, Spacer, Table, TableStyle, KeepTogether
)

# Optional: only mount Drive if running inside Google Colab.
try:
    from google.colab import drive, files  # noqa: F401
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# ---------------------------------------------------------------------------
# 0. Config -- EDIT THESE for your environment
# ---------------------------------------------------------------------------
# Single merged workbook containing BOTH sheets. All other sheets in this
# file are ignored.
WORKBOOK_FILE = "/content/drive/MyDrive/spring & summer.xlsx"

THESIS_SHEET_NAME = "Spring & Summer 2026"
EMPLOYEE_SHEET_NAME = "Employee ID"

REF_PREFIX = input("Enter Reference Prefix (e.g., UU/EEE/Thesis/OL/SU26/): ")
LETTER_DATE = input("Enter Letter Date (e.g., 10/05/2026): ")
SEMESTER_TEXT = input("Enter Semester Text (e.g., Summer 2026): ")
OUT_DIR = "/kaggle/working/letters" if os.path.isdir("/kaggle/working") else "letters"
FIXED_CR = 1.0
FIXED_HRS = 1.0

os.makedirs(OUT_DIR, exist_ok=True)


def s(x):
    """Trimmed string version of a cell value, '' for None/blank."""
    if x is None:
        return ""
    return str(x).strip()


def safe_filename(name):
    name = re.sub(r"[^a-zA-Z0-9]+", "_", name)
    name = re.sub(r"_+", "_", name).strip("_")
    return name


def clean_emp_id(x):
    """Normalize an Employee ID cell: openpyxl often reads e.g. 150271 as a
    float (150271.0) while IDs with leading zeros (e.g. '090104') are stored
    as text -- keep the latter as-is, strip '.0' off the former."""
    if x is None:
        return ""
    if isinstance(x, float) and x.is_integer():
        return str(int(x))
    return str(x).strip()


def normalize_name(n):
    """Case/whitespace/punctuation-insensitive key for name matching."""
    n = n.lower().strip()
    n = n.replace(".", "")


    n = re.sub(r"\s+", " ", n)
    return n


def strip_acronym(n):
    """Drop a trailing '(XYZ)' style acronym, e.g. 'Md. Rezaul Karim (RK)'
    -> 'Md. Rezaul Karim'."""
    return re.sub(r"\s*\([^)]*\)\s*$", "", n).strip()


# ---------------------------------------------------------------------------
# 1a. Build the Faculty Name -> Employee ID lookup from the 'Employee ID'
#     sheet of the SAME merged workbook. No other sheets are consulted.
#     Expected layout: header row 1, Employee ID in column A, Faculty Name
#     in column B, data starting row 2.
# ---------------------------------------------------------------------------
def load_employee_id_map(wb, sheet_name):
    emp_map = {}           # normalized name (full, incl. acronym) -> employee id
    emp_map_stripped = {}  # normalized name with acronym stripped -> employee id

    def add_entry(emp_id, name):
        emp_id = clean_emp_id(emp_id)
        name = s(name)
        if not emp_id or not name:
            return
        key_full = normalize_name(name)
        key_stripped = normalize_name(strip_acronym(name))
        emp_map.setdefault(key_full, emp_id)
        emp_map_stripped.setdefault(key_stripped, emp_id)

    if sheet_name not in wb.sheetnames:
        raise ValueError(
            f"Sheet '{sheet_name}' not found in workbook. "
            f"Available sheets: {wb.sheetnames}"
        )

    ws = wb[sheet_name]
    for row in ws.iter_rows(min_row=2, values_only=True):
        if len(row) < 2:
            continue
        emp_id_cell, name_cell = row[0], row[1]
        add_entry(emp_id_cell, name_cell)

    return emp_map, emp_map_stripped


def lookup_employee_id(faculty_name, emp_map, emp_map_stripped):
    key_full = normalize_name(faculty_name)
    if key_full in emp_map:
        return emp_map[key_full]
    key_stripped = normalize_name(strip_acronym(faculty_name))
    if key_stripped in emp_map_stripped:
        return emp_map_stripped[key_stripped]
    if key_stripped in emp_map:
        return emp_map[key_stripped]
    return ""


# ---------------------------------------------------------------------------
# Load the single merged workbook once, then pull both sheets from it.
# ---------------------------------------------------------------------------
wb = openpyxl.load_workbook(WORKBOOK_FILE, data_only=True)
print(f"Opened '{WORKBOOK_FILE}'. Sheets found: {wb.sheetnames} "
      f"(only '{THESIS_SHEET_NAME}' and '{EMPLOYEE_SHEET_NAME}' will be used).")

emp_map, emp_map_stripped = load_employee_id_map(wb, EMPLOYEE_SHEET_NAME)
print(f"Loaded {len(emp_map)} employee ID roster entries from sheet "
      f"'{EMPLOYEE_SHEET_NAME}'.")

# ---------------------------------------------------------------------------
# 1b. Locate the Thesis Distribution sheet (must exist; no silent fallback
#     to "first sheet" anymore, since we only want these two named sheets).
# ---------------------------------------------------------------------------
if THESIS_SHEET_NAME not in wb.sheetnames:
    raise ValueError(
        f"Sheet '{THESIS_SHEET_NAME}' not found in workbook. "
        f"Available sheets: {wb.sheetnames}"
    )
ws = wb[THESIS_SHEET_NAME]

# ---------------------------------------------------------------------------
# 2. Read & flatten the roster (forward-fill Group / Faculty across blank
#    continuation cells left by Excel's native merged cells)
# ---------------------------------------------------------------------------
rows = list(ws.iter_rows(min_row=2, values_only=True))

records = []
cur_group, cur_faculty = "", ""

for r in rows:
    # Defensive: pad short rows to 5 columns
    r = (list(r) + [None] * 5)[:5]
    slno, sid, name, batch, fac = r

    grp_str = s(slno)
    id_str = clean_emp_id(sid)
    name_str = s(name)
    batch_str = clean_emp_id(batch)
    fac_str = s(fac)

    if fac_str:
        cur_faculty = fac_str
    if grp_str:
        cur_group = grp_str

    # Rows with no Student ID are blank separator rows or section titles
    # (e.g. "Eve Batch 44 (...)") -- skip both.
    if not id_str or not cur_faculty:
        continue

    records.append({
        "Group": cur_group,
        "Faculty": cur_faculty,
        "ID": id_str,
        "Name": name_str,
        "Batch": batch_str,
    })

print(f"Parsed {len(records)} student rows.")

# ---------------------------------------------------------------------------
# 3. Collapse rows into groups, per faculty (preserve first-seen order)
# ---------------------------------------------------------------------------
faculty_order = []
faculty_map = {}

for rec in records:
    f = rec["Faculty"]
    if f not in faculty_map:
        faculty_map[f] = []
        faculty_order.append(f)

    groups = faculty_map[f]
    match = next((g for g in groups if g["GroupNo"] == rec["Group"]), None)
    if match is None:
        groups.append({
            "GroupNo": rec["Group"],
            "IDs": [rec["ID"]],
            "Names": [rec["Name"]],
            "Batch": [rec["Batch"]],
        })
    else:
        match["IDs"].append(rec["ID"])
        match["Names"].append(rec["Name"])
        match["Batch"].append(rec["Batch"])

print(f"Found {len(faculty_order)} faculty members.")

# Resolve Employee ID for every faculty member up front, and track misses.
faculty_emp_id = {}
unmatched_faculty = []
for f in faculty_order:
    emp_id = lookup_employee_id(f, emp_map, emp_map_stripped)
    faculty_emp_id[f] = emp_id
    if not emp_id:
        unmatched_faculty.append(f)

for f in faculty_order:
    groups = faculty_map[f]
    n_students = sum(len(g["IDs"]) for g in groups)
    emp_id_display = faculty_emp_id[f] or "NOT FOUND"
    print(f"  - {f:45s} {len(groups):2d} group(s), {n_students:3d} student(s)  "
          f"Emp ID: {emp_id_display}")

if unmatched_faculty:
    print(f"\nWARNING: could not match an Employee ID for {len(unmatched_faculty)} "
          f"faculty member(s) -- these letters will keep the '____' placeholder "
          f"for manual fill-in:")
    for f in unmatched_faculty:
        print(f"    - {f}")

# Sanity check: flag if grouping looks broken (groups == students)
one_to_one = sum(
    1 for f in faculty_order
    if len(faculty_map[f]) == sum(len(g["IDs"]) for g in faculty_map[f])
    and sum(len(g["IDs"]) for g in faculty_map[f]) > 1
)
if one_to_one > len(faculty_order) / 2:
    print(f"\nWARNING: {one_to_one}/{len(faculty_order)} faculty have "
          f"groups == students -- group detection may be broken. First 20 "
          f"raw rows of columns A/E:")
    for i, r in enumerate(rows[:20], start=1):
        r = (list(r) + [None] * 5)[:5]
        print(f"  row {i:3d} | col A = {s(r[0]) or '<blank>':20s} | "
              f"col E = {s(r[4]) or '<blank>':30s}")


# ---------------------------------------------------------------------------
# 4. PDF generation (one file per faculty)
# ---------------------------------------------------------------------------
def build_styles(table_font_size, body_font_size):
    return {
        "title": ParagraphStyle("title", fontName="Times-Bold", fontSize=12,
                                 alignment=TA_CENTER, leading=14),
        "title_u": ParagraphStyle("title_u", fontName="Times-Bold", fontSize=12,
                                   alignment=TA_CENTER, leading=14),
        "body": ParagraphStyle("body", fontName="Times-Roman", fontSize=body_font_size,
                                leading=body_font_size + 3),
        "cell": ParagraphStyle("cell", fontName="Times-Roman", fontSize=table_font_size,
                            alignment=TA_CENTER, leading=table_font_size),
        "cell_left": ParagraphStyle("cell_left", fontName="Times-Roman", fontSize=table_font_size,
                                alignment=TA_LEFT, leading=table_font_size),
        "cell_header": ParagraphStyle("cell_header", fontName="Times-Bold", fontSize=table_font_size,
                                  alignment=TA_CENTER, leading=table_font_size),
        "sig": ParagraphStyle("sig", fontName="Times-Bold", fontSize=body_font_size,
                               alignment=TA_CENTER, leading=body_font_size + 3),
    }


def generate_faculty_pdf(faculty_name, groups, emp_id, out_path):
    n_students = sum(len(g["IDs"]) for g in groups)

    if n_students <= 20:
        table_font_size, body_font_size = 9, 10
    elif n_students <= 30:
        table_font_size, body_font_size = 8, 9
    else:
        table_font_size, body_font_size = 7, 9

    styles = build_styles(table_font_size, body_font_size)

    doc = SimpleDocTemplate(
        out_path, pagesize=A4,
        leftMargin=0.6 * 72, rightMargin=0.6 * 72,
        topMargin=0.5 * 72, bottomMargin=0.5 * 72,
    )
    usable_width = doc.width

    story = []
    story.append(Paragraph("Department of Electrical and Electronic Engineering", styles["title"]))
    story.append(Paragraph("Uttara University", styles["title"]))
    story.append(Paragraph("<u>Claim of Thesis/Project Supervision</u>", styles["title_u"]))
    story.append(Spacer(1, 8))

    ref_table = Table(
        [[Paragraph(f"Ref: {REF_PREFIX}", styles["body"]),
          Paragraph(f"Date: {LETTER_DATE}", styles["body"])]],
        colWidths=[usable_width * 0.65, usable_width * 0.3],
    )
    ref_table.setStyle(TableStyle([
        ("BOX", (0, 0), (-1, -1), 0.75, colors.black),
        ("INNERGRID", (0, 0), (-1, -1), 0.75, colors.black),
        ("VALIGN", (0, 0), (-1, -1), "MIDDLE"),
        ("LEFTPADDING", (0, 0), (-1, -1), 4),
    ]))
    story.append(ref_table)
    story.append(Spacer(1, 8))

    story.append(Paragraph("To,", styles["body"]))
    emp_id_text = emp_id if emp_id else "____"
    story.append(Paragraph(
        f"{faculty_name}" + "&nbsp;" * 12 + f"Employee ID: {emp_id_text}", styles["body"]))
    story.append(Paragraph(
        f"Sub: Invitation letter to conduct Thesis/Project (EEE 4000) for {SEMESTER_TEXT}",
        styles["body"]))
    story.append(Spacer(1, 6))
    story.append(Paragraph("Dear Sir/Madam,", styles["body"]))
    story.append(Spacer(1, 6))

    intro = (
        "We are pleased to inform you that, you are nominated to conduct the course of "
        "&quot;Thesis/Project (EEE 4000)&quot; for " + SEMESTER_TEXT + ". The following "
        "students will be guided under your supervision in the concerned department as "
        "per above schedule."
    )
    story.append(Paragraph(intro, styles["body"]))
    story.append(Spacer(1, 8))

    # ---- Table: one physical row per STUDENT, with Sl.No/Group No/Cr./Hrs.
    #      genuinely merged (SPAN) across each group's row range.
    headers = ["Sl.No", "Group No", "Cr.", "Hrs.", "Student ID", "Student Name", "Batch"]
    table_data = [[Paragraph(h, styles["cell_header"]) for h in headers]]

    span_cmds = []
    total_cr = 0.0
    total_hrs = 0.0
    row_idx = 1  # row 0 is the header

    for g_idx, g in enumerate(groups, start=1):
        n_rows = len(g["IDs"])
        total_cr += FIXED_CR
        total_hrs += FIXED_HRS
        start_row = row_idx

        for r in range(n_rows):
            sl_txt = str(g_idx) if r == 0 else ""
            grp_txt = g["GroupNo"] if r == 0 else ""
            cr_txt = f"{FIXED_CR:.1f}" if r == 0 else ""
            hrs_txt = f"{FIXED_HRS:.1f}" if r == 0 else ""
            table_data.append([
                Paragraph(sl_txt, styles["cell"]),
                Paragraph(grp_txt, styles["cell"]),
                Paragraph(cr_txt, styles["cell"]),
                Paragraph(hrs_txt, styles["cell"]),
                Paragraph(g["IDs"][r], styles["cell"]),
                Paragraph(g["Names"][r].upper(), styles["cell_left"]),
                Paragraph(g["Batch"][r], styles["cell"]),
            ])
            row_idx += 1

        end_row = row_idx - 1
        if n_rows > 1:
            for col in range(4):  # Sl.No, Group No, Cr., Hrs.
                span_cmds.append(("SPAN", (col, start_row), (col, end_row)))
            span_cmds.append(("VALIGN", (0, start_row), (3, end_row), "MIDDLE"))

    # Total row
    table_data.append([
        Paragraph("<b>Total=</b>", styles["cell"]),
        Paragraph("", styles["cell_header"]),
        Paragraph(f"{total_cr:g}", styles["cell_header"]),
        Paragraph(f"{total_hrs:g}", styles["cell_header"]),
        Paragraph("", styles["cell"]),
        Paragraph("", styles["cell"]),
        Paragraph("", styles["cell"]),
    ])
    total_row_idx = row_idx
    span_cmds.append(("SPAN", (0, total_row_idx), (1, total_row_idx)))
    span_cmds.append(("ALIGN", (1, total_row_idx), (1, total_row_idx), "RIGHT"))
    span_cmds.append(("SPAN", (4, total_row_idx), (6, total_row_idx)))

    span_cmds.append(("LINEBELOW", (4, total_row_idx), (6, total_row_idx), 2, colors.white))
    span_cmds.append(("LINEBELOW", (0, total_row_idx), (1, total_row_idx), 2, colors.white))
    span_cmds.append(("LINEBEFORE", (0, total_row_idx), (1, total_row_idx), 1.1, colors.white))
    span_cmds.append(("LINEAFTER", (6, total_row_idx), (6, total_row_idx), 1.1, colors.white))
    span_cmds.append(("LINEBELOW", (2, total_row_idx), (3, total_row_idx), 1, colors.black))

    col_fractions = (0.08, 0.12, 0.08, 0.08, 0.14, 0.40, 0.10)
    col_widths = [usable_width*.90 * f for f in col_fractions]
    main_table = Table(table_data, colWidths=col_widths, repeatRows=1)
    style_cmds = [
        ("BOX", (0, 0), (-1, -1), 0.75, colors.black),
        ("INNERGRID", (0, 0), (-1, -1), 0.75, colors.black),
        ("VALIGN", (0, 0), (-1, -1), "MIDDLE"),
        ("ALIGN", (0, 0), (-1, -1), "CENTER"),
        ("LEFTPADDING", (0, 0), (-1, -1), 2),  # You can also reduce this
        ("RIGHTPADDING", (0, 0), (-1, -1), 2), # You can also reduce this
        ("TOPPADDING", (0, 0), (-1, -1), 1),    # Changed to 0
        ("BOTTOMPADDING", (0, 0), (-1, -1), 1), # Changed to 0
    ] + span_cmds
    main_table.setStyle(TableStyle(style_cmds))
    story.append(main_table)
    story.append(Spacer(1, 10))

    story.append(Paragraph("<b>Total Amount of Claim:&nbsp;&nbsp;Tk:</b>", styles["body"]))
    story.append(Spacer(1, 8))
    story.append(Paragraph("Thanking you very much for your cooperation.", styles["body"]))
    story.append(Paragraph("Yours Sincerely", styles["body"]))
    story.append(Spacer(1, 28))
    story.append(Paragraph("Thesis/Project Convener, Dept. of EEE, Uttara University", styles["body"]))
    story.append(Spacer(1, 28))

    sig_table = Table([[
        Paragraph("<br/>Signature of the Faculty Member<br/>(with date &amp; seal)", styles["sig"]),
        Paragraph("<br/>Signature of the Coordinator<br/>(with date &amp; seal)", styles["sig"]),
        Paragraph("<br/>Signature of the Chairman<br/>(with date &amp; seal)", styles["sig"]),
    ]], colWidths=[usable_width / 3] * 3)
    sig_table.setStyle(TableStyle([("VALIGN", (0, 0), (-1, -1), "TOP")]))
    story.append(sig_table)
    story.append(Spacer(1, 28))

    # Create the second signature table
    sig_table2 = Table([[
        Paragraph("Signature of the Deputy Registrar<br/>(with date &amp; seal)", styles["sig"]),
        Paragraph("Signature of the Deputy Registrar<br/>(with date &amp; seal)", styles["sig"]),
    ]], colWidths=[usable_width / 2] * 2)

    checked_by_section = [
        Paragraph("<b>Checked By:</b>", styles["body"]),
        Spacer(1, 28),
        sig_table2,
        Spacer(1, 28)
    ]

    story.append(KeepTogether(checked_by_section))

    doc.build(story)


# ---------------------------------------------------------------------------
# 5. Generate one PDF per faculty
# ---------------------------------------------------------------------------
for f in faculty_order:
    groups = faculty_map[f]
    n_students = sum(len(g["IDs"]) for g in groups)
    out_name = safe_filename(f) + ".pdf"
    out_path = os.path.join(OUT_DIR, out_name)

    generate_faculty_pdf(f, groups, faculty_emp_id[f], out_path)
    print(f"Generated: {out_name}  ({len(groups)} groups, {n_students} students)")

print(f"\nAll letters written to '{OUT_DIR}'.")
if unmatched_faculty:
    print(f"Reminder: {len(unmatched_faculty)} letter(s) still have a blank "
          f"Employee ID -- see the WARNING list printed above.")

if IN_COLAB:
    subprocess.check_call(["zip", "-r", "letters.zip", "letters/" if OUT_DIR == "letters" else OUT_DIR])
    files.download('letters.zip')